In [ ]:
# fix python path if working locally
from utils import fix_pythonpath_if_working_locally

fix_pythonpath_if_working_locally()

In [ ]:
from darts.datasets import ElectricityConsumptionZurichDataset
from darts.dataprocessing.transformers import Scaler

from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.utils.model_selection import train_test_split
import holidays
import numpy as np
from darts import TimeSeries

import pandas as pd
import numpy as np

import torch
#torch.set_float32_matmul_precision('highest')# 'medium' | 'high' | 'highest')

def split_series(series, train_end:str, val_end:str):
    train, remainder = series.split_after(train_end)
    val, test = remainder.split_after(val_end)

    return train, val, test

def scale_series(scaler, reference_series, additional_series=None):
    if not additional_series:
        additional_series = []
    transformed_reference = scaler.fit_transform(reference_series).astype(np.float32)
    transformed_additionals = [scaler.transform(additional).astype(np.float32) for additional in additional_series]

    return scaler, transformed_reference, *transformed_additionals


In [ ]:
# 1. load your series
series = ElectricityConsumptionZurichDataset().load()

# get just 5% of the data for demonstration purposes
# TODO REMOVE FOR REAL IMPLEMENTATION
series = series[:int(len(series) * 0.10)]
series.plot(label="Electricity Consumption in Zurich")

# 1. a) add covariates
#series = ElectricityConsumptionZurichDataset().load()
series = (
    series
    .add_holidays(country_code="CH", state="ZH")
    .stack(datetime_attribute_timeseries(series, attribute="month"))
    .stack(datetime_attribute_timeseries(series, attribute="weekday"))
    # exclude linear increase -> no trend in data (at least not visual)
)

all_column_names = series.columns.to_list()
print(f"All available columns are {all_column_names}")
target_series_names = ["Value_NE5",  "Value_NE7"]
selected_target_series = [target_series_names[0]]

covariates_names = ['Hr [%Hr]', 'RainDur [min]', 'StrGlo [W/m2]', 'T [°C]', 'WD [°]', 'WVs [m/s]', 'WVv [m/s]', 'p [hPa]']
future_covariates_names = ['holidays', 'month', 'weekday']


target_series = series[selected_target_series]
covariates = series[covariates_names]
future_covariates = series[future_covariates_names]

# 2. define split percentages
TRAIN_PERCENT = 0.65   # 65% for training
VAL_PERCENT = 0.15     # 15% for validation  
TEST_PERCENT = 0.2    # 20% for testing

# 3. calculate split points based on total length
total_len = len(series)
train_size = int(total_len * TRAIN_PERCENT)
val_size = int(total_len * VAL_PERCENT)

print(f"Split points: {[total_len, train_size, val_size]}")

# 4. get split timestamps
train_end = series.time_index[train_size - 1]
val_end = series.time_index[train_size + val_size - 1]

# 5. perform the splits
#train, remainder = series.split_after(train_end)
#val, test = remainder.split_after(val_end)

train, val, test = split_series(target_series, train_end=train_end, val_end=val_end)
covariate_train, covariate_val, covariate_test = split_series(covariates, train_end=train_end, val_end=val_end)

# future known covariates don't have to be split


print(f"Total series length: {total_len}")
print(f"Train size: {len(train)} ({len(train)/total_len*100:.1f}%)")
print(f"Val size: {len(val)} ({len(val)/total_len*100:.1f}%)")
print(f"Test size: {len(test)} ({len(test)/total_len*100:.1f}%)")
print(f"train: {train.time_index[0]} → {train.time_index[-1]}")
print(f"val:   {val.time_index[0]} → {val.time_index[-1]}")
print(f"test:  {test.time_index[0]} → {test.time_index[-1]}")

# fix erronous data
# TODO

# scale the series
target_scaler, train_scaled, val_scaled, test_scaled = scale_series(Scaler(), train, [val, test])
covariate_scaler, train_cov_scaled, val_cov_scaled, test_cov_scaled = scale_series(Scaler(), covariate_train, [covariate_val, covariate_test])
future_covariates_scaler, future_covariates_scaled = scale_series(Scaler(), future_covariates)


In [ ]:
# From some reason the model has to be initialized like this
# when we use darts.model import TFT it has a pytroch error, so we need to avoid it with this import
from darts.models.forecasting.tft_model import TFTModel
import optuna
import torch
from pytorch_lightning.callbacks import EarlyStopping, Callback
from optuna.integration import PyTorchLightningPruningCallback
from darts.metrics import smape
from darts.utils.likelihood_models import QuantileRegression

# Patch pruning callback (due to compatibility issue)
class PatchedPruningCallback(PyTorchLightningPruningCallback, Callback):
    pass

In [ ]:
from darts.models import TFTSSMModel

# default quantiles for QuantileRegression
quantiles = [
    0.01,
    0.05,
    0.1,
    0.15,
    0.2,
    0.25,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.75,
    0.8,
    0.85,
    0.9,
    0.95,
    0.99,
]
input_chunk_length = 24
forecast_horizon = 12
n_epochs = 3


my_ssm_model = TFTSSMModel(
    input_chunk_length=input_chunk_length,
    output_chunk_length=forecast_horizon,
    hidden_size=64,
    lstm_layers=2,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=32,
    n_epochs=n_epochs,
    add_relative_index=False,
    add_encoders=None,
    likelihood=QuantileRegression(
        quantiles=quantiles
    ),  # QuantileRegression is set per default
    # loss_fn=MSELoss(),
    random_state=42,
    mamba_native = False,
    log_tensorboard=True,
)

my_ssm_model.fit(
    series=train_scaled, 
    past_covariates=train_cov_scaled, 
    future_covariates=future_covariates_scaled, 
    verbose=True,
    val_series=val_scaled,
    val_past_covariates=val_cov_scaled,
    val_future_covariates=future_covariates_scaled,
    )

In [ ]:
future_covariates.start_time()

In [ ]:
covariates.end_time()

In [ ]:
pred = my_ssm_model.predict(
    n=12,
    series=test_scaled[:-forecast_horizon].astype(np.float32), 
    #series=target_scaler.transform(target_series),
    past_covariates=covariate_scaler.transform(covariates).astype(np.float32), 
    future_covariates=future_covariates.astype(np.float32)
    )
smape(test_scaled, pred)

In [ ]:
import matplotlib.pyplot as plt

figsize = (9, 6)
lowest_q, low_q, high_q, highest_q = 0.01, 0.1, 0.9, 0.99
label_q_outer = f"{int(lowest_q * 100)}-{int(highest_q * 100)}th percentiles"
label_q_inner = f"{int(low_q * 100)}-{int(high_q * 100)}th percentiles"

def eval_model(model, n, actual_series, past_covariates, future_covariates):
    #shortened_series = 
    pred_series = model.predict(
        n=n, 
        num_samples=n,
        series=actual_series,
        past_covariates=past_covariates,
        future_covariates=future_covariates
        )


    # plot actual series
    plt.figure(figsize=figsize)
    actual_series[: pred_series.end_time()].plot(label="actual")

    # plot prediction with quantile ranges
    pred_series.plot(
        low_quantile=lowest_q, high_quantile=highest_q, label=label_q_outer
    )
    pred_series.plot(low_quantile=low_q, high_quantile=high_q, label=label_q_inner)


    print(len(actual_series), len(pred_series))
    plt.title(f"SMAPE: {smape(actual_series, pred_series):.2f}%")
    plt.legend()


eval_model(
    model=my_ssm_model, 
    n=500, 
    actual_series=test_scaled[:-forecast_horizon].astype(np.float32), 
    past_covariates=covariate_scaler.transform(covariates)[:-forecast_horizon].astype(np.float32),
    future_covariates=future_covariates.astype(np.float32)[:-forecast_horizon],
    )